In [0]:
from pyspark.sql.session import SparkSession
print(spark)#already instantiated by databricks
spark1=SparkSession.builder.getOrCreate()
print(spark1)#we instantiated

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS telecom_catalog_assign;
CREATE SCHEMA IF NOT EXISTS telecom_catalog_assign.landing_zone;
CREATE VOLUME IF NOT EXISTS telecom_catalog_assign.landing_zone.landing_vol;


In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/")


In [0]:
customer_csv = """101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
"""

usage_tsv = """customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
"""

tower_logs_region1 = """event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
"""


In [0]:
base_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"

dbutils.fs.put(f"{base_path}/customer/customer.csv", customer_csv, overwrite=True)
dbutils.fs.put(f"{base_path}/usage/usage.tsv", usage_tsv, overwrite=True)
dbutils.fs.put(f"{base_path}/tower/tower_region1.psv", tower_logs_region1, overwrite=True)


In [0]:
df_customer_raw = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "true")
    .csv(f"{base_path}/customer/customer.csv")
)


In [0]:
df_customer = df_customer_raw.toDF(
    "customer_id", "customer_name", "age", "city", "plan_type"
)

display(df_customer)


In [0]:
df_customer = df_customer_raw.toDF(
    "customer_id", "customer_name", "age", "city", "plan_type"
)

display(df_customer)


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType


In [0]:
tower_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("tower_id", StringType(), True),
    StructField("signal_strength", IntegerType(), True),
    StructField("timestamp", TimestampType(), True)
])


In [0]:
df_tower = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .schema(tower_schema)
    .csv(f"{base_path}/tower/tower_region1.psv")
)

display(df_tower)


In [0]:
# Source (staging / temp location)
src_base = "/tmp/telecom_staging"

# Target Volume base path
tgt_base = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"


In [0]:
dbutils.fs.put(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",
    customer_csv,
    overwrite=True
)


In [0]:
dbutils.fs.put(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv",
    usage_tsv,
    overwrite=True
)


In [0]:
dbutils.fs.mkdirs(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1"
)

dbutils.fs.put(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_region1.psv",
    tower_logs_region1,
    overwrite=True
)


In [0]:
dbutils.fs.mkdirs(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2"
)


In [0]:
base_tower_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower"


In [0]:
df_glob = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .option("pathGlobFilter", "*.psv")
    .csv(f"{base_tower_path}/region1")
)

display(df_glob)


In [0]:
df_multi_path = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .csv([
        f"{base_tower_path}/region1",
        f"{base_tower_path}/region2"
    ])
)

display(df_multi_path)


In [0]:
df_recursive = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .option("recursiveFileLookup", "true")
    .csv(base_tower_path)
)

display(df_recursive)


In [0]:
cust_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv"
usage_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv"


In [0]:
df_cust_raw = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "false")
    .csv(cust_path)
)

df_cust_raw.printSchema()
display(df_cust_raw)


In [0]:
df_usage_raw = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "false")
    .option("sep", "\t")
    .csv(usage_path)
)

display(df_usage_raw)


In [0]:
df_cust_inf = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(cust_path)
)

df_cust_inf.printSchema()
display(df_cust_inf)


In [0]:
df_usage_inf = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", "\t")
    .csv(usage_path)
)

df_usage_inf.printSchema()
display(df_usage_inf)


In [0]:
cust_path  = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv"
usage_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv"
tower_path = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1"


In [0]:
df_customer_raw = (
    spark.read
    .option("header", "false")
    .option("inferSchema", "false")
    .csv(cust_path)
)


In [0]:
df_customer = df_customer_raw.toDF(
    "customer_id",
    "customer_name",
    "age",
    "city",
    "plan_type"
)

df_customer.printSchema()
display(df_customer)


In [0]:
usage_schema = "customer_id INT, voice_mins INT, data_mb INT, sms_count INT"


In [0]:
df_usage = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .schema(usage_schema)
    .csv(usage_path)
)

df_usage.printSchema()
display(df_usage)


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, TimestampType
)


In [0]:
tower_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("tower_id", StringType(), True),
    StructField("signal_strength", IntegerType(), True),
    StructField("event_timestamp", TimestampType(), True)
])


In [0]:
df_tower = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .schema(tower_schema)
    .csv(tower_path)
)

df_tower.printSchema()
display(df_tower)


In [0]:
base_vol = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"


In [0]:
df_customer.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{base_vol}/customer_csv_out")


In [0]:
df_usage.write \
    .mode("append") \
    .option("header", "true") \
    .csv(f"{base_vol}/usage_csv_out")


In [0]:
df_tower.write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", "|") \
    .csv(f"{base_vol}/tower_csv_out")


In [0]:
df_tower_read = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .csv(f"{base_vol}/tower_csv_out")
)

df_tower_read.show(5)


In [0]:
dbutils.fs.ls(f"{base_vol}/customer_csv_out")


In [0]:
base_vol = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"


In [0]:
df_customer.write \
    .mode("overwrite") \
    .json(f"{base_vol}/customer_json_out")


In [0]:
df_usage.write \
    .mode("append") \
    .option("compression", "snappy") \
    .json(f"{base_vol}/usage_json_out")


In [0]:
df_tower.write \
    .mode("ignore") \
    .json(f"{base_vol}/tower_json_out")


In [0]:
df_tower_read = spark.read.json(f"{base_vol}/tower_json_out")
df_tower_read.show(5)


In [0]:
dbutils.fs.ls(f"{base_vol}/tower_json_out")


In [0]:
base_vol = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"


In [0]:
df_customer.write \
    .mode("overwrite") \
    .option("compression", "gzip") \
    .parquet(f"{base_vol}/customer_parquet_out")


In [0]:
df_tower.write \
    .mode("overwrite") \
    .option("compression", "gzip") \
    .parquet(f"{base_vol}/tower_parquet_out")


In [0]:
df_usage_read = spark.read.parquet(f"{base_vol}/usage_parquet_out")
df_usage_read.show(5)


In [0]:
df_usage.write \
    .mode("overwrite") \
    .parquet(f"{base_vol}/usage_parquet_out")


In [0]:
df_usage_read = spark.read.parquet(f"{base_vol}/usage_parquet_out")
df_usage_read.show(5)


In [0]:
base_vol = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"


In [0]:
df_customer.write \
    .mode("overwrite") \
    .orc(f"{base_vol}/customer_orc_out")


In [0]:
df_tower.write \
    .mode("overwrite") \
    .orc(f"{base_vol}/tower_orc_out")


In [0]:
dbutils.fs.ls(f"{base_vol}/tower_orc_out")


In [0]:
base_vol = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"


In [0]:
df_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{base_vol}/customer_delta")


In [0]:
df_usage.write \
    .format("delta") \
    .mode("append") \
    .save(f"{base_vol}/usage_delta")


In [0]:
df_tower.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{base_vol}/tower_delta")


In [0]:
dbutils.fs.ls(f"{base_vol}/tower_delta")


In [0]:
df_usage_read = spark.read \
    .format("delta") \
    .load(f"{base_vol}/usage_delta")

df_usage_read.show(5)


In [0]:
df_customer.write \
    .format("delta") \
    .saveAsTable("telecom_catalog_assign.landing_zone.customer_mng")


In [0]:
df_usage.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("telecom_catalog_assign.landing_zone.usage_mng")


In [0]:
%sql
DROP TABLE telecom_catalog_assign.landing_zone.customer_mng;


In [0]:
%sql SHOW TABLES IN telecom_catalog_assign.landing_zone


In [0]:
spark.sql("""
SELECT * 
FROM telecom_catalog_assign.landing_zone.usage_mng
""").show()


In [0]:
spark.sql("""
SELECT customer_id, voice_mins
FROM telecom_catalog_assign.landing_zone.usage_mng
WHERE voice_mins > 100
""").show()


In [0]:
spark.sql("""
SELECT 
    COUNT(*) AS total_customers,
    SUM(data_mb) AS total_data_usage
FROM telecom_catalog_assign.landing_zone.usage_mng
""").show()


In [0]:
%sql
-- Run in %sql cell
CREATE TABLE telecom_catalog_assign.landing_zone.customer_insert_tbl (
  customer_id INT,
  customer_name STRING,
  age STRING,
  city STRING,
  plan_type STRING
)
USING DELTA;


In [0]:
df_customer.write \
    .insertInto("telecom_catalog_assign.landing_zone.customer_insert_tbl")


In [0]:
%sql
-- Run in %sql cell
CREATE TABLE telecom_catalog_assign.landing_zone.usage_insert_tbl (
  customer_id INT,
  voice_mins INT,
  data_mb INT,
  sms_count INT
)
USING DELTA;


In [0]:
df_usage.write \
    .mode("overwrite") \
    .insertTable("telecom_catalog_assign.landing_zone.usage_insert_tbl")


In [0]:
df_usage.write \
    .mode("overwrite") \
    .saveAsTable("telecom_catalog_assign.landing_zone.usage_insert_tbl")


In [0]:
df_usage.write \
    .insertInto("telecom_catalog_assign.landing_zone.usage_insert_tbl")


In [0]:
base_vol = "/Volumes/telecom_catalog_assign/landing_zone/landing_vol"


In [0]:
df_customer.write \
    .format("xml") \
    .option("rowTag", "cust") \
    .mode("overwrite") \
    .save(f"{base_vol}/customer_xml_out")


In [0]:
dbutils.fs.ls(f"{base_vol}/customer_xml_out")
dbutils.fs.ls(f"{base_vol}/usage_xml_out")


In [0]:
spark.read.format("xml")


In [0]:
df_orc = spark.read.orc(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer_orc_out"
)


In [0]:
df_orc.write \
    .mode("overwrite") \
    .parquet("dbfs:/FileStore/orc_to_parquet_output")


In [0]:
df_orc.write \
    .mode("overwrite") \
    .parquet("s3://your-bucket/path/orc_to_parquet/")

In [0]:
df_parquet = spark.read.parquet(
    "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer_parquet_out"
)


In [0]:
df_parquet.write \
    .format("delta") \
    .mode("overwrite") \
    .save("dbfs:/FileStore/parquet_to_delta")
